# Attention Metrics (Counts)
**1. Attention Analytics**  
Objective: Quantify contestant attention in Reddit discussion.
Pipeline:
- Aggregate islander_windows by target_islander and EPISODE.
- Calculate raw mention counts and seasonal percentage share.
- Visualize "Attention Spikes" using line charts mapped to villa events.
- Extract top 3 most frequently used emojis per Islander to identify "archetypal" reaction.
Source: heavy-cleaned `islander_windows` from 03. Counts are mention windows, not unique comments or people. Sentiment uses the separate light-text table in 05.


## Preview Data

In [ ]:
import duckdb

import pandas as pd
pd.set_option("display.max_colwidth", None)

In [ ]:
con = duckdb.connect("../data/love_island_reddit.duckdb")

top_10_windows = con.execute("""
SELECT *
FROM islander_windows
LIMIT 10
""").df()

top_10_windows

In [ ]:
random_10_windows = con.execute("""
SELECT *
FROM islander_windows
USING SAMPLE 10 ROWS
""").df()

random_10_windows

## Counting Windows

### Attention Windows Per Episode

In [ ]:
episode_windows = con.execute("""
SELECT
    w.episode,
    COUNT(*) AS num_windows
FROM islander_windows AS w
WHERE w.keep_analysis = 'Y'
  AND w.episode IS NOT NULL
GROUP BY w.episode
ORDER BY w.episode
""").df()

#### Line Graph

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.figure(figsize=(15, 6))

sns.lineplot(
    data=episode_windows,
    x="episode",
    y="num_windows",
    marker="o",
    linewidth=2,
)

# Important Season 8 Events
recoupling_eps = [5, 13, 27]
casa_start = 18
casa_recoupling = 21
aftersun_episodes = [10, 16, 22, 28, 34]

# Regular recouplings (red dashed)
for ep in recoupling_eps:
    plt.axvline(
        x=ep,
        color="tab:red",
        linestyle="--",
        linewidth=1.5,
        alpha=0.7,
    )

# Aftersun episodes (gray dotted)
for ep in aftersun_episodes:
    plt.axvline(
        x=ep,
        color="0.6",
        linestyle=":",
        linewidth=1.5,
        alpha=0.6,
    )

# Casa Amor begins (orange dashed)
plt.axvline(
    x=casa_start,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    alpha=0.8,
)

# Casa Amor recoupling (red dashed)
plt.axvline(
    x=casa_recoupling,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    alpha=0.8,
)

# Labels
ymax = plt.ylim()[1]

# Recoupling labels
for ep in recoupling_eps:
    plt.text(
        ep,
        ymax * 0.97,
        "Recoupling",
        rotation=90,
        va="top",
        ha="right",
        color="tab:red",
        fontsize=10,
    )

# Aftersun labels
for ep in aftersun_episodes:
    plt.text(
        ep,
        ymax * 0.97,
        "Aftersun",
        rotation=90,
        va="top",
        ha="right",
        color="0.4",
        fontsize=12,
    )

# Casa Amor labels
plt.text(
    casa_start,
    ymax * 0.97,
    "Casa Amor Begins",
    rotation=90,
    va="top",
    ha="right",
    color="tab:orange",
    fontsize=12,
)

plt.text(
    casa_recoupling,
    ymax * 0.97,
    "Casa Recoupling",
    rotation=90,
    va="top",
    ha="right",
    color="tab:orange",
    fontsize=12,
)

# Formatting
plt.title("Islander Attention by Episode")
plt.xlabel("Episode")
plt.ylabel("Number of Windows")
plt.xticks(range(1, 36))

plt.tight_layout()
plt.show()

#### Bar Chart

In [ ]:
aftersun_episodes = [10, 16, 22, 28, 34]

plt.figure(figsize=(15, 6))

sns.barplot(
    data=episode_windows,
    x="episode",
    y="num_windows",
    color="steelblue",
)

for episode in aftersun_episodes:
    plt.axvspan(
        episode - 1.5,
        episode - 0.5,
        color="orange",
        alpha=0.35,
    )

plt.title("Islander Attention by Episode")
plt.xlabel("Episode")
plt.ylabel("Number of Windows")
plt.tight_layout()
plt.show()

Episodes 10, 16, 22, 28, and 34 are all Aftersun episodes.

### Attention Windows Per Islander Per Episode

In [ ]:
episode_islander_windows = con.execute("""
SELECT
    w.episode,
    w.target_islander,
    COUNT(*) AS num_windows
FROM islander_windows AS w
WHERE w.keep_analysis = 'Y'
  AND w.episode IS NOT NULL
GROUP BY
    w.episode,
    w.target_islander
ORDER BY
    w.episode,
    num_windows DESC
""").df()

In [ ]:
episode_islander_windows.head(10)

### Top 3 Islanders Per Episode By Window Count

In [ ]:
top_3_episode_islanders = episode_islander_windows \
    .sort_values(
        ["episode", "num_windows"],
        ascending=[True, False]
    ) \
    .groupby("episode") \
    .head(3) \
    .reset_index(drop=True)

In [ ]:
episode_islander_windows["attention_share"] = (
    episode_islander_windows["num_windows"]
    / episode_islander_windows
        .groupby("episode")["num_windows"]
        .transform("sum")
)

In [ ]:
plot_df = (
    top_3_episode_islanders
    .copy()
    .sort_values(["episode", "num_windows"], ascending=[True, False])
)

plot_df["rank"] = (
    plot_df
    .groupby("episode")
    .cumcount()
    .add(1)
)

colors = {
    1: "#d62728",
    2: "#1f77b4",
    3: "#7f7f7f",
}

fig, ax = plt.subplots(figsize=(18, 9))

for _, row in plot_df.iterrows():
    ax.text(
        row["episode"],
        row["num_windows"],
        row["target_islander"],
        color=colors[row["rank"]],
        fontsize=8,
        fontweight="bold" if row["rank"] == 1 else "normal",
        ha="center",
        va="center",
    )

ax.set_xlim(0, 36)
ax.set_ylim(
    0,
    plot_df["num_windows"].max() * 1.08,
)

ax.set_xticks(range(1, 36))
ax.set_title("Top Three Most-Mentioned Islanders per Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("Number of Windows")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Top 3 Islanders Per Episode By Attention Share

In [ ]:
top_k_episode_islanders = (
    episode_islander_windows
    .copy()
    .assign(
        discussion_pct=lambda x:
            x["num_windows"]
            / x.groupby("episode")["num_windows"].transform("sum")
    )
    .sort_values(
        ["episode", "num_windows"],
        ascending=[True, False],
    )
)

top_k_episode_islanders["rank"] = (
    top_k_episode_islanders
    .groupby("episode")
    .cumcount()
    .add(1)
)

top_k_episode_islanders = (
    top_k_episode_islanders
    .query("rank <= 3")
    .reset_index(drop=True)
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(24, 9))

rank_colors = sns.color_palette(
    "flare",
    n_colors=top_k_episode_islanders["rank"].nunique()
)

for _, row in top_k_episode_islanders.iterrows():
    ax.text(
        row["episode"],
        row["discussion_pct"],
        row["target_islander"],
        ha="center",
        va="center",
        fontsize=7,
        color=rank_colors[int(row["rank"]) - 1],
        fontweight="bold" if row["rank"] == 1 else "normal",
    )

# Required because there are no plotted points to set the limits
ax.set_xlim(0.5, 35.5)
ax.set_ylim(
    0,
    top_k_episode_islanders["discussion_pct"].max() * 1.1,
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xticks(range(1, 36))

ax.set_title("Top Islanders’ Share of Discussion by Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("Percentage of Islander Discussion")

plt.tight_layout()
plt.show()

This is so we can repel the names from intersecting on plot.

In [ ]:
from adjustText import adjust_text
from matplotlib.lines import Line2D

rank_colors = {
    1: "#d73027",
    2: "#fc8d59",
    3: "#4575b4",
    4: "#74add1",
    5: "#999999",
}

fig, ax = plt.subplots(figsize=(24, 9))

texts = []

for _, row in top_k_episode_islanders.iterrows():
    rank = int(row["rank"])

    texts.append(
        ax.text(
            row["episode"],
            row["discussion_pct"],
            row["target_islander"],
            color=rank_colors[rank],
            fontweight="bold" if rank == 1 else "normal",
            ha="center",
            va="center",
            fontsize=7,
        )
    )

ax.set_xlim(0.5, 35.5)
ax.set_ylim(
    0,
    top_k_episode_islanders["discussion_pct"].max() * 1.15,
)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xticks(range(1, 36))

ax.set_title("Top Islanders’ Share of Discussion by Episode")
ax.set_xlabel("Episode")
ax.set_ylabel("Percentage of Islander Discussion")

adjust_text(
    texts,
    ax=ax,
    only_move={
        "text": "y",
        "static": "y",
        "explode": "y",
        "pull": "y",
    },
    expand=(1.1, 1.3),
    force_text=(0.2, 0.5),
    arrowprops={
        "arrowstyle": "-",
        "color": "gray",
        "alpha": 0.35,
        "lw": 0.5,
    },
)

legend_items = [
    Line2D(
        [0],
        [0],
        color=rank_colors[rank],
        marker="o",
        linestyle="",
        label=f"Rank {rank}",
    )
    for rank in sorted(rank_colors)
    if rank <= top_k_episode_islanders["rank"].max()
]

ax.legend(
    handles=legend_items,
    title="Episode rank",
    loc="upper right",
)

plt.tight_layout()
plt.show()

### Attention Share Grid

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Calculate total windows using ALL islanders, not only the top three
episode_totals = (
    episode_islander_windows
    .groupby("episode", as_index=False)["num_windows"]
    .sum()
    .rename(columns={"num_windows": "total_windows"})
)

plot_df = (
    top_3_episode_islanders
    .copy()
    .merge(
        episode_totals,
        on="episode",
        how="left",
    )
    .assign(
        discussion_share=lambda x:
            x["num_windows"] / x["total_windows"]
    )
    .sort_values(
        ["episode", "num_windows"],
        ascending=[True, False],
    )
)

plot_df["rank"] = (
    plot_df
    .groupby("episode")
    .cumcount()
    .add(1)
)

colors = {
    1: "#d62728",
    2: "#1f77b4",
    3: "#7f7f7f",
}

In [ ]:
def format_islander_label(name, share):
    name_parts = name.split(maxsplit=1)

    if len(name_parts) == 2:
        formatted_name = f"{name_parts[0]} {name_parts[1]}"
    else:
        formatted_name = name_parts[0]

    return f"{formatted_name}\n{share:.1%}"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 24))

for _, row in plot_df.iterrows():
    label = format_islander_label(
        row["target_islander"],
        row["discussion_share"],
    )

    ax.text(
        row["rank"],
        row["episode"],
        label,
        color=colors[row["rank"]],
        fontsize=15,
        fontweight="bold" if row["rank"] == 1 else "normal",
        ha="center",
        va="center",
        linespacing=1.15,
    )

ax.set_xlim(0.5, 3.5)
ax.set_ylim(35.5, 0.5)  # episode 1 at the top

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["Rank 1", "Rank 2", "Rank 3"])

ax.set_yticks(range(1, 36))

ax.set_title("Top Three Most-Discussed Islanders per Episode")
ax.set_xlabel("Attention Rank")
ax.set_ylabel("Episode")

ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", alpha=0.15)

plt.tight_layout()
plt.show()

### Percent Change in Attention Share
Changes are reported only for adjacent episode numbers; gaps remain missing rather than being treated as consecutive observations.


In [ ]:
episode_islander_windows = (
    episode_islander_windows
    .sort_values(["target_islander", "episode"])
    .reset_index(drop=True)
)

episode_islander_windows["share_change_pp"] = (
    episode_islander_windows
    .groupby("target_islander")["attention_share"]
    .diff()
    .mul(100)
)

episode_islander_windows["share_pct_change"] = (
    episode_islander_windows
    .groupby("target_islander")["attention_share"]
    .pct_change(fill_method=None)
    .mul(100)
)
# Missing episodes must not be described as a one-episode change.
previous_episode = episode_islander_windows.groupby("target_islander")["episode"].shift()
consecutive = episode_islander_windows["episode"].sub(previous_episode).eq(1)
episode_islander_windows.loc[
    ~consecutive, ["share_change_pp", "share_pct_change"]
] = float("nan")
episode_islander_windows["share_pct_change"] = episode_islander_windows["share_pct_change"].replace(
    [float("inf"), -float("inf")], float("nan")
)


In [ ]:
change_matrix = (
    episode_islander_windows
    .pivot(
        index="target_islander",
        columns="episode",
        values="share_change_pp",
    )
)

plt.figure(figsize=(20, 10))

sns.heatmap(
    change_matrix,
    cmap="RdBu_r",
    center=0,
    linewidths=0.2,
    cbar_kws={
        "label": "Change from previous episode (percentage points)"
    },
)

plt.title("Change in Islander Attention Share by Episode")
plt.xlabel("Episode")
plt.ylabel("Islander")
plt.tight_layout()
plt.show()

### Overall Attention

In [ ]:
overall_islander_windows = (
    episode_islander_windows
    .groupby("target_islander", as_index=False)["num_windows"]
    .sum()
    .sort_values("num_windows", ascending=False)
)

overall_islander_windows["attention_share"] = (
    overall_islander_windows["num_windows"]
    / overall_islander_windows["num_windows"].sum()
)

In [ ]:
overall_islander_windows

In [ ]:
con.close()

In [ ]:
from datetime import datetime
print(datetime.now())

## Emoji + Image Analysis

In [ ]:
with duckdb.connect("../data/love_island_reddit.duckdb") as con:
    df_windows = con.execute("""
        SELECT comment_id, target_islander, window_text, episode
        FROM islander_windows
        WHERE keep_analysis = 'Y' AND episode IS NOT NULL
    """).df()


In [ ]:
df_windows.head(3)

In [ ]:
# Image markers are counted separately from emojis.
image_counts = (
    df_windows.assign(image_count=df_windows["window_text"].str.count(r"\[img\]"))
    .groupby("target_islander", as_index=False)["image_count"].sum()
)
image_counts


In [ ]:
import re
import emoji

def extract_emojis(text):
    if pd.isna(text):
        return []
    names = re.findall(r"\[emoji_([^\[\]]+)\]", text)
    result = []
    for name in names:
        shortcode = f":{name}:"
        restored = emoji.emojize(shortcode, language="en")
        if restored != shortcode:
            result.append(restored)
    return result

# Audit tags damaged by heavy cleaning or unsupported by the emoji library.
raw_emoji_tags = df_windows["window_text"].str.findall(r"\[emoji_([^\[\]]+)\]").explode().dropna()
unrecognized_emoji_tags = raw_emoji_tags[
    raw_emoji_tags.map(lambda name: emoji.emojize(f":{name}:", language="en") == f":{name}:")
].value_counts().rename_axis("tag").reset_index(name="count")
print(f"Excluded {unrecognized_emoji_tags['count'].sum():,} unrecognized emoji occurrences.")


# One row per emoji occurrence in an islander's window
emoji_long = (
    df_windows[["comment_id", "target_islander", "window_text", "episode"]]
    .assign(emoji=lambda df: df["window_text"].apply(extract_emojis))
    .explode("emoji")
    .dropna(subset=["emoji"])
)

# Count each emoji per islander, then select their top three
top_3_emojis = (
    emoji_long
    .groupby(["target_islander", "emoji"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["target_islander", "count", "emoji"],
        ascending=[True, False, True],
    )
    .groupby("target_islander")
    .head(3)
)

# Format a summary table
islander_emoji_table = (
    top_3_emojis
    .assign(
        emoji_with_count=lambda df:
            df["emoji"] + " (" + df["count"].astype(str) + ")"
    )
    .groupby("target_islander", as_index=False)
    .agg(top_3_emojis=("emoji_with_count", ", ".join))
)

islander_emoji_table

Lift is the emoji's share for a specific islander divided by the share for all other islanders. If for example crying eyes represents 10% of Trinity's emojis and it represents 2% of everyone else's, then crying eyes is 5 times as common for Trinity than others.
Minimum occurrences: 10 overall, 5 per episode; minimum lift: 2. These are exploratory filters. Infinite lift means an emoji occurs only for this islander in the comparison pool, not a measured finite effect. Unrecognized tags are excluded and listed separately.


In [ ]:
overall_emoji_metrics = (
    emoji_long.groupby(["target_islander", "emoji"])
    .size()
    .reset_index(name="count")
)

overall_emoji_metrics["islander_total"] = overall_emoji_metrics.groupby(
    "target_islander"
)["count"].transform("sum")

overall_emoji_metrics["emoji_total"] = overall_emoji_metrics.groupby(
    "emoji"
)["count"].transform("sum")

total = overall_emoji_metrics["count"].sum()

overall_emoji_metrics["share"] = overall_emoji_metrics["count"] / overall_emoji_metrics["islander_total"]

overall_emoji_metrics["others_share"] = (
    (overall_emoji_metrics["emoji_total"] - overall_emoji_metrics["count"])
    / (total - overall_emoji_metrics["islander_total"]).replace(0, float("nan"))
)

# No comparison observations => undefined lift; zero comparison share => infinite lift.
overall_emoji_metrics["lift"] = overall_emoji_metrics["share"] / overall_emoji_metrics["others_share"]
overall_emoji_metrics["exclusive_emoji"] = overall_emoji_metrics["others_share"].eq(0)

standout_emojis = (
    overall_emoji_metrics.query("count >= 10 and lift >= 2")
    .sort_values(
        ["target_islander", "lift", "count"],
        ascending=[True, False, False],
    )
    .groupby("target_islander")
    .head(3)
    [["target_islander", "emoji", "count", "share", "lift"]]
    .reset_index(drop=True)
)

standout_emojis.round({"share": 3, "lift": 2})

In [ ]:
standout_emoji_table = (
    standout_emojis
    .groupby("target_islander", as_index=False)
    .agg(emojis=("emoji", " ".join))
    .rename(columns={"target_islander": "Islander", "emojis": "Standout emojis"})
)

standout_emoji_table

Now, let's do by episode.

In [ ]:
episode_emoji_metrics = (
    emoji_long
    .groupby(["episode", "target_islander", "emoji"])
    .size()
    .reset_index(name="count")
)

episode_emoji_metrics["islander_total"] = episode_emoji_metrics.groupby(
    ["episode", "target_islander"]
)["count"].transform("sum")

episode_emoji_metrics["emoji_total"] = episode_emoji_metrics.groupby(
    ["episode", "emoji"]
)["count"].transform("sum")

episode_emoji_metrics["episode_total"] = episode_emoji_metrics.groupby(
    "episode"
)["count"].transform("sum")

episode_emoji_metrics["share"] = episode_emoji_metrics["count"] / episode_emoji_metrics["islander_total"]

others_total = episode_emoji_metrics["episode_total"] - episode_emoji_metrics["islander_total"]
episode_emoji_metrics["others_share"] = (
    (episode_emoji_metrics["emoji_total"] - episode_emoji_metrics["count"])
    / others_total.where(others_total > 0)
)

# No comparison observations => undefined lift; zero comparison share => infinite lift.
episode_emoji_metrics["lift"] = episode_emoji_metrics["share"] / episode_emoji_metrics["others_share"]
episode_emoji_metrics["exclusive_emoji"] = episode_emoji_metrics["others_share"].eq(0)

episode_standouts = (
    episode_emoji_metrics.query("count >= 5 and lift >= 2")
    .sort_values(
        ["episode", "target_islander", "lift", "count", "emoji"],
        ascending=[True, True, False, False, True],
    )
    .groupby(["episode", "target_islander"])
    .head(3)
)

episode_emoji_table = (
    episode_standouts
    .groupby(["episode", "target_islander"], as_index=False)
    .agg(emojis=("emoji", " ".join))
    .rename(columns={
        "episode": "Episode",
        "target_islander": "Islander",
        "emojis": "Standout emojis",
    })
)

episode_emoji_table

## Save dashboard tables to DuckDB

DuckDB is the primary dashboard store. All tables below are refreshed together in the `dashboard` schema. Full emoji metrics support filtering and tooltips; compact tables support display. The final section also exports CSV copies.


In [ ]:
dashboard_tables = {
    "episode_windows": episode_windows,
    "episode_islander_windows": episode_islander_windows,
    "overall_islander_windows": overall_islander_windows,
    "top_k_episode_islanders": top_k_episode_islanders,
    "standout_emoji_table": standout_emoji_table,
    "episode_emoji_table": episode_emoji_table,
    "standout_emojis": standout_emojis,
    "episode_standouts": episode_standouts,
    "islander_emoji_table": islander_emoji_table,
    "top_3_emojis": top_3_emojis,
    "overall_emoji_metrics": overall_emoji_metrics,
    "episode_emoji_metrics": episode_emoji_metrics,
    "image_counts": image_counts,
    "unrecognized_emoji_tags": unrecognized_emoji_tags,
}


In [ ]:
assert episode_windows["num_windows"].sum() == len(df_windows)
assert episode_islander_windows["num_windows"].sum() == len(df_windows)
assert overall_islander_windows["num_windows"].sum() == len(df_windows)
assert episode_islander_windows.groupby("episode")["attention_share"].sum().sub(1).abs().lt(1e-9).all()

with duckdb.connect("../data/love_island_reddit.duckdb") as con:
    con.execute("CREATE SCHEMA IF NOT EXISTS dashboard")
    con.execute("BEGIN TRANSACTION")
    try:
        for name, table in dashboard_tables.items():
            con.register("export_df", table)
            con.execute(f"CREATE OR REPLACE TABLE dashboard.{name} AS SELECT * FROM export_df")
            assert con.execute(f"SELECT COUNT(*) FROM dashboard.{name}").fetchone()[0] == len(table)
            con.unregister("export_df")
        con.execute("COMMIT")
    except Exception:
        con.execute("ROLLBACK")
        raise
print(f"Saved and verified {len(dashboard_tables)} tables in the dashboard schema.")

## Export dashboard tables to CSV
Run after the dashboard tables are created. Saves UTF-8 CSV copies alongside the DuckDB tables; rerunning replaces the CSV files.

In [ ]:
from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
export_dir = project_root / "tables" / "attention_dashboard"
export_dir.mkdir(parents=True, exist_ok=True)

for name, table in dashboard_tables.items():
    table.to_csv(export_dir / f"{name}.csv", index=False, encoding="utf-8")

print(f"Saved {len(dashboard_tables)} CSV files to {export_dir}")
